# 01b - PPO with critic warm-start and K=2

This notebook keeps the exact-answer PPO baseline from `01_ppo.ipynb` while
changing only critic training:

1. The critic warms up on 64 separate GSM8K training problems before the policy
   receives any gradient update.
2. Every policy update is followed by exactly two full critic forward/backward
   updates (`K=2`), matching the faster-value-update cadence studied by SAO.

The actor objective, KL-shaped token rewards, GAE, value clipping, 1,024 PPO
training problems, 256 held-out evaluation problems, and evaluation protocol
remain the same as PPO 1.

## Hyperparameters

The warm-start set is disjoint from both policy training and official-test
evaluation. Two PPO policy epochs are retained; `K=2` applies independently
inside each policy epoch.

In [ ]:
# Model and data
sft_checkpoint = "ppbhatt500/rl-ablations-sft-2026-07-17"
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_revision = "740312add88f781978c0658806c59bc2815b9866"
seed = 17

# Data sizes
critic_warmstart_problems = 64
train_problems = 1024
eval_problems = 256
num_epochs = 1

# PPO and sequence sizes
batch_size = 32
forward_batch_size = 4
num_ppo_epochs = 2
critic_updates_per_policy_update = 2
max_prompt_tokens = 384
max_completion_tokens = 256

# Optimization
temperature = 0.8
top_p = 1.0
actor_learning_rate = 1e-6
critic_learning_rate = 2e-6
weight_decay = 0.0
policy_clip_epsilon = 0.2
value_clip_epsilon = 0.2
kl_coefficient = 0.05
gamma = 1.0
gae_lambda = 0.95
max_grad_norm = 1.0

# Evaluation and outputs
eval_every_steps = 8
eval_batch_size = 16
wandb_project = "swe-rl-ablations"
wandb_run_name = "ppo-2"
output_path = "./checkpoints/ppo_2"

optimizer_steps = train_problems // batch_size
warmstart_batches = critic_warmstart_problems // batch_size
assert critic_warmstart_problems == 64
assert train_problems == 1024 and eval_problems == 256
assert batch_size == 32 and forward_batch_size == 4
assert batch_size % forward_batch_size == 0
assert critic_warmstart_problems % batch_size == 0
assert num_ppo_epochs == 2 and critic_updates_per_policy_update == 2
print(
    f"PPO 2: {warmstart_batches} warm-start batches, "
    f"{optimizer_steps} policy batches, K={critic_updates_per_policy_update}"
)

## Imports and reproducibility

In [ ]:
from pathlib import Path
import json
import os
import re
import time

import torch
import torch.nn as nn
import wandb
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, set_seed
from trl.experimental.ppo import PPOConfig, PPOTrainer
from trl.trainer.utils import selective_log_softmax

project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
checkpoint_path = sft_checkpoint
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is required to load the private SFT checkpoint.")
os.environ["WANDB_PROJECT"] = wandb_project
set_seed(seed)

## Separate warm-start, policy-training, and evaluation rows

The shuffled official training split supplies 64 critic-only warm-start rows
followed by 1,024 different PPO rows. Evaluation remains on the official test
split.

In [ ]:
all_train_rows = load_dataset(
    dataset_id,
    dataset_config,
    split="train",
    revision=dataset_revision,
).shuffle(seed=seed).select(range(critic_warmstart_problems + train_problems))
warmstart_rows = all_train_rows.select(range(critic_warmstart_problems))
train_rows = all_train_rows.select(
    range(critic_warmstart_problems, critic_warmstart_problems + train_problems)
)
eval_rows = load_dataset(
    dataset_id,
    dataset_config,
    split="test",
    revision=dataset_revision,
).shuffle(seed=seed).select(range(eval_problems))

warmstart_questions = set(warmstart_rows["question"])
train_questions = set(train_rows["question"])
eval_questions = set(eval_rows["question"])
if warmstart_questions & train_questions:
    raise RuntimeError("Critic warm-start rows overlap PPO policy-training rows.")
if (warmstart_questions | train_questions) & eval_questions:
    raise RuntimeError("Official-train rows overlap official-test evaluation rows.")
print(
    f"Critic warm-start: {len(warmstart_rows)} | "
    f"PPO train: {len(train_rows)} | held-out eval: {len(eval_rows)}"
)

## Exact-answer reward

In [ ]:
def gold_number(answer):
    match = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Malformed GSM8K answer: {answer!r}")
    return match.group(1).replace(",", "")


def predicted_number(completion):
    match = re.search(
        r"Final answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$",
        completion.strip(),
        re.IGNORECASE,
    )
    return None if not match else match.group(1).replace(",", "")


def exact_reward(completion, answer):
    prediction = predicted_number(completion)
    if prediction is None:
        return -1.0
    return 1.0 if float(prediction) == float(gold_number(answer)) else 0.0

## Atomic JSON and tensor diagnostics

In [ ]:
def write_json(path, document, indent=2):
    temporary_path = path.with_suffix(".json.tmp")
    separators = (",", ":") if indent is None else None
    temporary_path.write_text(
        json.dumps(document, indent=indent, separators=separators, allow_nan=False) + "\n"
    )
    temporary_path.replace(path)


def tensor_stats(values, mask):
    selected = values[mask.bool()].detach().float()
    return {
        "count": int(selected.numel()),
        "mean": float(selected.mean().item()),
        "std": float(selected.std(unbiased=False).item()),
        "min": float(selected.min().item()),
        "max": float(selected.max().item()),
    }


def masked_rows(values, mask, digits=7):
    rows = []
    for row_values, row_mask in zip(
        values.detach().float().cpu(),
        mask.detach().bool().cpu(),
    ):
        rows.append([round(float(value), digits) for value in row_values[row_mask].tolist()])
    return rows


def masked_mean(values, mask):
    return (values * mask).sum() / mask.sum().clamp_min(1)


def gradient_norm(parameters):
    norms = [
        parameter.grad.detach().norm(2)
        for parameter in parameters
        if parameter.grad is not None
    ]
    return float(torch.norm(torch.stack(norms), 2).item()) if norms else 0.0


def explained_variance(prediction, target, mask):
    prediction = prediction[mask.bool()]
    target = target[mask.bool()]
    variance = target.var(unbiased=False)
    if variance <= 1e-12:
        return 0.0
    return float((1 - (target - prediction).var(unbiased=False) / variance).item())

## Tokenizer, prompt-length guard, actor, reference, and critic

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint_path,
    use_fast=True,
    token=hf_token,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def prompt_text(question):
    messages = [
        {
            "role": "user",
            "content": (
                "Solve this grade-school math problem. Show your reasoning, then "
                "end with `Final answer: <number>`.\n\n" + question
            ),
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


all_questions = (
    list(warmstart_rows["question"])
    + list(train_rows["question"])
    + list(eval_rows["question"])
)
prompt_lengths = [
    len(input_ids)
    for input_ids in tokenizer(
        [prompt_text(question) for question in all_questions],
        truncation=False,
    )["input_ids"]
]
if max(prompt_lengths) > max_prompt_tokens:
    raise RuntimeError(
        f"Prompt truncation required: max={max(prompt_lengths)}, limit={max_prompt_tokens}"
    )
print(f"Maximum prompt length: {max(prompt_lengths)} tokens")

actor = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    token=hf_token,
)
reference = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    token=hf_token,
)
critic = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_path,
    num_labels=1,
    ignore_mismatched_sizes=True,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    token=hf_token,
)
critic.config.pad_token_id = tokenizer.pad_token_id
for parameter in reference.parameters():
    parameter.requires_grad_(False)
reference.eval()

## Rollout generation

In [ ]:
def generate_rollout(model, rows):
    prompts = [prompt_text(question) for question in rows["question"]]
    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_prompt_tokens,
    ).to(model.device)
    prompt_width = encoded.input_ids.shape[1]
    generated = model.generate(
        **encoded,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_completion_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    completion = generated[:, prompt_width:]
    if completion.shape[1] < max_completion_tokens:
        padding = torch.full(
            (len(rows), max_completion_tokens - completion.shape[1]),
            tokenizer.pad_token_id,
            device=completion.device,
        )
        completion = torch.cat([completion, padding], dim=1)
    eos = completion.eq(tokenizer.eos_token_id)
    first_eos = torch.where(
        eos.any(1),
        eos.float().argmax(1),
        torch.full((len(rows),), max_completion_tokens - 1, device=completion.device),
    ).long()
    positions = torch.arange(max_completion_tokens, device=completion.device).unsqueeze(0)
    generated_mask = positions <= first_eos.unsqueeze(1)
    input_ids = torch.cat([encoded.input_ids, completion], dim=1)
    attention_mask = torch.cat([encoded.attention_mask, generated_mask.long()], dim=1)
    completion_mask = torch.zeros(
        (len(rows), input_ids.shape[1] - 1),
        device=model.device,
    )
    completion_mask[:, prompt_width - 1:] = generated_mask.float()
    completions = tokenizer.batch_decode(completion, skip_special_tokens=True)
    return input_ids, attention_mask, completion_mask, completions

## Chunked policy and critic forward passes

In [ ]:
def token_logprobs(model, input_ids, attention_mask, requires_grad):
    chunks = []
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        for start in range(0, len(input_ids), forward_batch_size):
            ids = input_ids[start:start + forward_batch_size]
            mask = attention_mask[start:start + forward_batch_size]
            logits = model(
                input_ids=ids,
                attention_mask=mask,
                use_cache=False,
            ).logits[:, :-1]
            chunks.append(selective_log_softmax(logits, ids[:, 1:]))
    return torch.cat(chunks)


def token_values(model, input_ids, attention_mask, requires_grad):
    chunks = []
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        for start in range(0, len(input_ids), forward_batch_size):
            ids = input_ids[start:start + forward_batch_size]
            mask = attention_mask[start:start + forward_batch_size]
            output = model(
                input_ids=ids,
                attention_mask=mask,
                output_hidden_states=True,
                use_cache=False,
                return_dict=True,
            )
            chunks.append(model.score(output.hidden_states[-1][:, :-1]).squeeze(-1).float())
    return torch.cat(chunks)

## Monte Carlo warm-start targets and PPO GAE

Warm-start targets are backward cumulative token rewards and do not use critic
predictions. Normal PPO then returns to the original fixed-λ GAE.

In [ ]:
def monte_carlo_returns(token_rewards, mask):
    returns = torch.zeros_like(token_rewards)
    running = torch.zeros(len(token_rewards), device=token_rewards.device)
    for index in range(token_rewards.shape[1] - 1, -1, -1):
        running = (token_rewards[:, index] + gamma * running) * mask[:, index]
        returns[:, index] = running
    return returns


def full_gae(token_rewards, values, mask):
    deltas = torch.zeros_like(values)
    advantages = torch.zeros_like(values)
    running = torch.zeros(len(values), device=values.device)
    for index in range(values.shape[1] - 1, -1, -1):
        next_value = (
            values[:, index + 1] * mask[:, index + 1]
            if index + 1 < values.shape[1]
            else 0.0
        )
        delta = (
            token_rewards[:, index] + gamma * next_value - values[:, index]
        ) * mask[:, index]
        deltas[:, index] = delta
        running = (delta + gamma * gae_lambda * running) * mask[:, index]
        advantages[:, index] = running
    raw_advantages = advantages.clone()
    returns = raw_advantages + values
    selected = raw_advantages[mask.bool()]
    advantages = (
        (raw_advantages - selected.mean())
        / (selected.std(unbiased=False) + 1e-8)
    ) * mask
    return advantages, returns, deltas, raw_advantages

## Reward construction shared by warm-start and PPO

In [ ]:
@torch.no_grad()
def rollout_targets(policy, reference_model, value_model, rows, use_gae):
    input_ids, attention_mask, completion_mask, completions = generate_rollout(
        policy,
        rows,
    )
    rewards = torch.tensor(
        [
            exact_reward(completion, answer)
            for completion, answer in zip(completions, rows["answer"])
        ],
        device=policy.device,
    )
    old_logps = token_logprobs(
        policy,
        input_ids,
        attention_mask,
        requires_grad=False,
    )
    reference_logps = token_logprobs(
        reference_model,
        input_ids,
        attention_mask,
        requires_grad=False,
    )
    old_values = token_values(
        value_model,
        input_ids,
        attention_mask,
        requires_grad=False,
    )
    token_rewards = -kl_coefficient * (
        old_logps - reference_logps
    ) * completion_mask
    final_positions = (
        torch.arange(completion_mask.shape[1], device=policy.device).unsqueeze(0)
        * completion_mask.long()
    ).max(1).values
    token_rewards[
        torch.arange(len(rewards), device=policy.device),
        final_positions,
    ] += rewards
    if use_gae:
        advantages, returns, deltas, raw_advantages = full_gae(
            token_rewards,
            old_values,
            completion_mask,
        )
    else:
        returns = monte_carlo_returns(token_rewards, completion_mask)
        deltas = None
        raw_advantages = returns - old_values
        selected = raw_advantages[completion_mask.bool()]
        advantages = (
            (raw_advantages - selected.mean())
            / (selected.std(unbiased=False) + 1e-8)
        ) * completion_mask
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "completion_mask": completion_mask,
        "completions": completions,
        "rewards": rewards,
        "old_logps": old_logps,
        "reference_logps": reference_logps,
        "old_values": old_values,
        "token_rewards": token_rewards,
        "advantages": advantages,
        "returns": returns,
        "deltas": deltas,
        "raw_advantages": raw_advantages,
    }

## One PPO policy update

In [ ]:
def update_policy(
    policy,
    optimizer,
    rollout,
):
    mask = rollout["completion_mask"]
    indices = torch.randperm(len(mask), device=policy.device)
    token_count = mask[indices].sum()
    token_count_value = float(token_count.item())
    optimizer.zero_grad(set_to_none=True)
    loss_total = 0.0
    clipped_token_count = 0.0
    started_at = time.perf_counter()

    # Microbatches accumulate one logical batch-32 policy gradient.
    for start in range(0, len(indices), forward_batch_size):
        selected = indices[start:start + forward_batch_size]
        selected_mask = mask[selected]
        new_logps = token_logprobs(
            policy,
            rollout["input_ids"][selected],
            rollout["attention_mask"][selected],
            requires_grad=True,
        )
        ratio = torch.exp(new_logps - rollout["old_logps"][selected])
        unclipped = ratio * rollout["advantages"][selected]
        clipped = ratio.clamp(
            1 - policy_clip_epsilon,
            1 + policy_clip_epsilon,
        ) * rollout["advantages"][selected]
        numerator = -(
            torch.minimum(unclipped, clipped) * selected_mask
        ).sum()
        (numerator / token_count).backward()
        loss_total += float(numerator.detach().item())
        controlling_clip = (
            (rollout["advantages"][selected] > 0)
            & (ratio > 1 + policy_clip_epsilon)
        ) | (
            (rollout["advantages"][selected] < 0)
            & (ratio < 1 - policy_clip_epsilon)
        )
        clipped_token_count += float(
            (controlling_clip.float() * selected_mask).sum().item()
        )
    grad_norm = gradient_norm(policy.parameters())
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
    optimizer.step()
    torch.cuda.synchronize()
    return {
        "loss": loss_total / token_count_value,
        "grad_norm": grad_norm,
        "clipped_fraction": clipped_token_count / token_count_value,
        "seconds": time.perf_counter() - started_at,
    }

## One full critic forward/backward update

In [ ]:
def update_critic(
    value_model,
    optimizer,
    rollout,
    clip_values,
):
    mask = rollout["completion_mask"]
    indices = torch.randperm(len(mask), device=mask.device)
    token_count = mask[indices].sum()
    token_count_value = float(token_count.item())
    values_before = torch.zeros_like(rollout["old_values"])
    optimizer.zero_grad(set_to_none=True)
    loss_total = 0.0
    started_at = time.perf_counter()

    # This loop is one full critic pass. The caller invokes it K=2 times after
    # every policy update, so each invocation has its own zero_grad/backward/step.
    for start in range(0, len(indices), forward_batch_size):
        selected = indices[start:start + forward_batch_size]
        selected_mask = mask[selected]
        new_values = token_values(
            value_model,
            rollout["input_ids"][selected],
            rollout["attention_mask"][selected],
            requires_grad=True,
        )
        values_before[selected] = new_values.detach()
        if clip_values:
            clipped_values = rollout["old_values"][selected] + (
                new_values - rollout["old_values"][selected]
            ).clamp(-value_clip_epsilon, value_clip_epsilon)
            squared_error = torch.maximum(
                (new_values - rollout["returns"][selected]).square(),
                (clipped_values - rollout["returns"][selected]).square(),
            )
        else:
            # Warm-start has no trusted old critic, so clipping would anchor the
            # new critic to random initial predictions. Use ordinary MSE instead.
            squared_error = (
                new_values - rollout["returns"][selected]
            ).square()
        numerator = 0.5 * (squared_error * selected_mask).sum()
        (numerator / token_count).backward()
        loss_total += float(numerator.detach().item())
    grad_norm = gradient_norm(value_model.parameters())
    torch.nn.utils.clip_grad_norm_(value_model.parameters(), max_grad_norm)
    optimizer.step()
    torch.cuda.synchronize()
    return {
        "loss": loss_total / token_count_value,
        "grad_norm": grad_norm,
        "seconds": time.perf_counter() - started_at,
        "value_stats_before": tensor_stats(values_before, mask),
        "error_stats_before": tensor_stats(
            values_before - rollout["returns"],
            mask,
        ),
        "values_before": masked_rows(values_before, mask),
        "errors_before": masked_rows(
            values_before - rollout["returns"],
            mask,
        ),
    }

## Critic-only warm-start over 64 separate problems

In [ ]:
def warmstart_critic(
    policy,
    reference_model,
    value_model,
    critic_optimizer,
    run,
    gae_document,
):
    warmstart_metrics = []
    for batch_index, start in enumerate(
        range(0, len(warmstart_rows), batch_size),
        start=1,
    ):
        rows = warmstart_rows.select(
            range(start, min(start + batch_size, len(warmstart_rows)))
        )
        rollout = rollout_targets(
            policy,
            reference_model,
            value_model,
            rows,
            use_gae=False,
        )
        trace = {
            "batch": batch_index,
            "problem_start": start,
            "problem_stop": start + len(rows),
            "target_type": "monte_carlo",
            "reward_mean": float(rollout["rewards"].mean().item()),
            "initial_value_stats": tensor_stats(
                rollout["old_values"],
                rollout["completion_mask"],
            ),
            "return_stats": tensor_stats(
                rollout["returns"],
                rollout["completion_mask"],
            ),
            "critic_updates": [],
            "examples": [],
        }
        token_id_rows = masked_rows(
            rollout["input_ids"][:, 1:],
            rollout["completion_mask"],
        )
        initial_value_rows = masked_rows(
            rollout["old_values"],
            rollout["completion_mask"],
        )
        token_reward_rows = masked_rows(
            rollout["token_rewards"],
            rollout["completion_mask"],
        )
        return_rows = masked_rows(
            rollout["returns"],
            rollout["completion_mask"],
        )
        for row_index in range(len(rows)):
            trace["examples"].append(
                {
                    "problem_index": start + row_index,
                    "question": rows[row_index]["question"],
                    "completion": rollout["completions"][row_index],
                    "reward": float(rollout["rewards"][row_index].item()),
                    "token_ids": token_id_rows[row_index],
                    "token_rewards": token_reward_rows[row_index],
                    "monte_carlo_returns": return_rows[row_index],
                    "initial_values": initial_value_rows[row_index],
                }
            )

        # K=2 is also used during warm-start: two separate optimizer steps on
        # each 32-problem batch, for four warm-start steps across 64 problems.
        for critic_update_index in range(critic_updates_per_policy_update):
            result = update_critic(
                value_model,
                critic_optimizer,
                rollout,
                clip_values=False,
            )
            trace["critic_updates"].append(
                {
                    "update": critic_update_index + 1,
                    **result,
                }
            )
        final_values = token_values(
            value_model,
            rollout["input_ids"],
            rollout["attention_mask"],
            requires_grad=False,
        )
        final_ev = explained_variance(
            final_values,
            rollout["returns"],
            rollout["completion_mask"],
        )
        trace["final_value_stats"] = tensor_stats(
            final_values,
            rollout["completion_mask"],
        )
        trace["final_explained_variance"] = final_ev
        final_value_rows = masked_rows(
            final_values,
            rollout["completion_mask"],
        )
        for example, values in zip(trace["examples"], final_value_rows):
            example["final_values"] = values
        gae_document["warmstart"].append(trace)
        metrics = {
            "warmstart/batch": batch_index,
            "warmstart/reward": float(rollout["rewards"].mean().item()),
            "warmstart/critic_loss": sum(
                item["loss"] for item in trace["critic_updates"]
            ) / len(trace["critic_updates"]),
            "warmstart/critic_explained_variance": final_ev,
        }
        warmstart_metrics.append(metrics)
        run.log(metrics)
    return warmstart_metrics

## Compact per-rollout GAE trace

In [ ]:
def build_gae_trace(step, problem_start, rows, rollout):
    mask = rollout["completion_mask"]
    active_token_ids = rollout["input_ids"][:, 1:]
    trace = {
        "step": step,
        "problem_start": problem_start,
        "problem_stop": problem_start + len(rows),
        "stats": {
            "old_values": tensor_stats(rollout["old_values"], mask),
            "token_rewards": tensor_stats(rollout["token_rewards"], mask),
            "deltas": tensor_stats(rollout["deltas"], mask),
            "raw_advantages": tensor_stats(rollout["raw_advantages"], mask),
            "normalized_advantages": tensor_stats(rollout["advantages"], mask),
            "returns": tensor_stats(rollout["returns"], mask),
        },
        "examples": [],
        "policy_epochs": [],
    }
    token_rows = [
        row[row_mask].detach().cpu().tolist()
        for row, row_mask in zip(active_token_ids, mask.bool())
    ]
    fields = {
        "old_logprobs": masked_rows(rollout["old_logps"], mask),
        "reference_logprobs": masked_rows(rollout["reference_logps"], mask),
        "old_values": masked_rows(rollout["old_values"], mask),
        "token_rewards": masked_rows(rollout["token_rewards"], mask),
        "deltas": masked_rows(rollout["deltas"], mask),
        "raw_advantages": masked_rows(rollout["raw_advantages"], mask),
        "normalized_advantages": masked_rows(rollout["advantages"], mask),
        "returns": masked_rows(rollout["returns"], mask),
    }
    for row_index in range(len(rows)):
        example = {
            "problem_index": problem_start + row_index,
            "question": rows[row_index]["question"],
            "completion": rollout["completions"][row_index],
            "reward": float(rollout["rewards"][row_index].item()),
            "token_ids": token_rows[row_index],
        }
        for name, values in fields.items():
            example[name] = values[row_index]
        trace["examples"].append(example)
    return trace

## Deterministic held-out evaluation

In [ ]:
@torch.no_grad()
def exact_evaluate(policy):
    policy.eval()
    rewards = []
    for start in range(0, len(eval_rows), eval_batch_size):
        stop = min(start + eval_batch_size, len(eval_rows))
        rows = eval_rows.select(range(start, stop))
        prompts = [prompt_text(question) for question in rows["question"]]
        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to(policy.device)
        generated = policy.generate(
            **encoded,
            do_sample=False,
            max_new_tokens=max_completion_tokens,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
        completions = tokenizer.batch_decode(
            generated[:, encoded.input_ids.shape[1]:],
            skip_special_tokens=True,
        )
        rewards.extend(
            exact_reward(completion, answer)
            for completion, answer in zip(completions, rows["answer"])
        )
    policy.train()
    return sum(rewards) / len(rewards)

## TRL construction adapter

In [ ]:
class UnusedNeuralReward(nn.Module):
    """TRL requires a reward module; decoded exact rewards replace it."""

    def __init__(self):
        super().__init__()
        self.anchor = nn.Parameter(torch.zeros(()), requires_grad=False)


unused_reward_model = UnusedNeuralReward()

## Initialize metrics, optimizers, W&B, and critic warm-start

In [ ]:
def initialize_training(trainer):
    wrapped = trainer.accelerator.unwrap_model(trainer.model)
    policy = wrapped.policy
    value_model = wrapped.value_model
    policy.train()
    value_model.train()
    policy_optimizer = torch.optim.AdamW(
        policy.parameters(),
        lr=actor_learning_rate,
        weight_decay=weight_decay,
    )
    critic_optimizer = torch.optim.AdamW(
        value_model.parameters(),
        lr=critic_learning_rate,
        weight_decay=weight_decay,
    )
    output_dir = (project_root / output_path).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "metrics.json"
    gae_path = output_dir / "gae.json"
    metrics_document = {
        "base_checkpoint": str(checkpoint_path),
        "critic_warmstart_problems": critic_warmstart_problems,
        "critic_updates_per_policy_update": critic_updates_per_policy_update,
        "train_problems": len(trainer.exact_train_rows),
        "eval_problems": len(eval_rows),
        "batch_size": batch_size,
        "ppo_policy_epochs": num_ppo_epochs,
        "steps": [],
    }
    gae_document = {
        "schema_version": 2,
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "critic_warmstart_problems": critic_warmstart_problems,
        "critic_updates_per_policy_update": critic_updates_per_policy_update,
        "warmstart": [],
        "updates": [],
    }
    run = wandb.init(project=wandb_project, name=wandb_run_name)
    torch.cuda.reset_peak_memory_stats()

    # Warm-start changes only the critic, so step zero is still the common SFT policy.
    initial_eval = exact_evaluate(policy)
    metrics_document["steps"].append({"step": 0, "eval/reward": initial_eval})
    run.log({"eval/reward": initial_eval}, step=0)
    warmstart_critic(
        policy,
        trainer.ref_model,
        value_model,
        critic_optimizer,
        run,
        gae_document,
    )
    write_json(metrics_path, metrics_document)
    write_json(gae_path, gae_document, indent=None)
    return {
        "policy": policy,
        "value_model": value_model,
        "policy_optimizer": policy_optimizer,
        "critic_optimizer": critic_optimizer,
        "output_dir": output_dir,
        "metrics_path": metrics_path,
        "gae_path": gae_path,
        "metrics_document": metrics_document,
        "gae_document": gae_document,
        "run": run,
        "training_started": time.perf_counter(),
        "last_eval_reward": initial_eval,
    }

## Optimize one 32-problem PPO batch

In [ ]:
def optimize_training_batch(trainer, state, global_step, start):
    rows = trainer.exact_train_rows.select(range(start, start + batch_size))
    rollout_started = time.perf_counter()
    rollout = rollout_targets(
        state["policy"],
        trainer.ref_model,
        state["value_model"],
        rows,
        use_gae=True,
    )
    rollout_seconds = time.perf_counter() - rollout_started
    trace = build_gae_trace(global_step, start, rows, rollout)
    policy_results = []
    critic_results = []

    # Every policy step is followed immediately by K=2 full critic optimizer steps.
    for policy_epoch in range(num_ppo_epochs):
        policy_result = update_policy(
            state["policy"],
            state["policy_optimizer"],
            rollout,
        )
        policy_results.append(policy_result)
        epoch_trace = {
            "policy_epoch": policy_epoch + 1,
            "policy": policy_result,
            "critic_updates": [],
        }
        for critic_update_index in range(critic_updates_per_policy_update):
            critic_result = update_critic(
                state["value_model"],
                state["critic_optimizer"],
                rollout,
                clip_values=True,
            )
            critic_results.append(critic_result)
            epoch_trace["critic_updates"].append(
                {"update": critic_update_index + 1, **critic_result}
            )
        trace["policy_epochs"].append(epoch_trace)

    final_values = token_values(
        state["value_model"],
        rollout["input_ids"],
        rollout["attention_mask"],
        requires_grad=False,
    )
    post_update_logps = token_logprobs(
        state["policy"],
        rollout["input_ids"],
        rollout["attention_mask"],
        requires_grad=False,
    )
    trace["final_critic"] = {
        "value_stats": tensor_stats(final_values, rollout["completion_mask"]),
        "error_stats": tensor_stats(
            final_values - rollout["returns"],
            rollout["completion_mask"],
        ),
        "values": masked_rows(final_values, rollout["completion_mask"]),
    }
    return {
        "rollout": rollout,
        "rollout_seconds": rollout_seconds,
        "trace": trace,
        "policy_results": policy_results,
        "critic_results": critic_results,
        "final_values": final_values,
        "post_update_logps": post_update_logps,
    }

## Evaluate, persist traces, and log one completed batch

In [ ]:
def persist_training_batch(state, global_step, result, step_started):
    rollout = result["rollout"]
    state["gae_document"]["updates"].append(result["trace"])
    state["gae_document"]["summary"] = {
        "completed": False,
        "updates_completed": global_step,
        "problems_completed": global_step * batch_size,
    }
    write_json(state["gae_path"], state["gae_document"], indent=None)

    evaluation_seconds = 0.0
    eval_reward = None
    if global_step % eval_every_steps == 0 or global_step == optimizer_steps:
        evaluation_started = time.perf_counter()
        eval_reward = exact_evaluate(state["policy"])
        evaluation_seconds = time.perf_counter() - evaluation_started
        state["last_eval_reward"] = eval_reward
    elapsed_seconds = time.perf_counter() - state["training_started"]
    policy_results = result["policy_results"]
    critic_results = result["critic_results"]
    metrics = {
        "progress/problems_completed": global_step * batch_size,
        "progress/updates_completed": global_step,
        "train/reward": float(rollout["rewards"].mean().item()),
        "loss/policy": sum(x["loss"] for x in policy_results) / len(policy_results),
        "objective/kl_sft": float(
            masked_mean(
                result["post_update_logps"] - rollout["reference_logps"],
                rollout["completion_mask"],
            ).item()
        ),
        "policy/tokens_clipped_or_masked_pct": 100 * sum(
            x["clipped_fraction"] for x in policy_results
        ) / len(policy_results),
        "critic/loss": sum(x["loss"] for x in critic_results) / len(critic_results),
        "critic/explained_variance": explained_variance(
            result["final_values"],
            rollout["returns"],
            rollout["completion_mask"],
        ),
        "grad_norm/policy": sum(x["grad_norm"] for x in policy_results) / len(policy_results),
        "grad_norm/critic": sum(x["grad_norm"] for x in critic_results) / len(critic_results),
        "time/policy_forward_backward_s": sum(x["seconds"] for x in policy_results),
        "time/critic_forward_backward_s": sum(x["seconds"] for x in critic_results),
        "time/rollout_s": result["rollout_seconds"],
        "time/evaluation_s": evaluation_seconds,
        "time/step_s": time.perf_counter() - step_started,
        "time/elapsed_s": elapsed_seconds,
        "time/eta_s": elapsed_seconds / global_step * (optimizer_steps - global_step),
        "gpu/peak_memory_gb": torch.cuda.max_memory_allocated() / 1024**3,
    }
    if eval_reward is not None:
        metrics["eval/reward"] = eval_reward
    state["metrics_document"]["steps"].append({"step": global_step, **metrics})
    state["metrics_document"]["summary"] = {
        "completed": False,
        "updates_completed": global_step,
        "problems_completed": global_step * batch_size,
        "latest_eval_reward": state["last_eval_reward"],
        "latest_train_reward": metrics["train/reward"],
        "elapsed_s": elapsed_seconds,
        "eta_s": metrics["time/eta_s"],
    }
    write_json(state["metrics_path"], state["metrics_document"])
    state["run"].log(metrics, step=warmstart_batches + global_step)

## PPO 2 training orchestration

In [ ]:
def run_ppo2_training(trainer):
    state = initialize_training(trainer)
    for global_step, start in enumerate(
        range(0, len(trainer.exact_train_rows), batch_size),
        start=1,
    ):
        step_started = time.perf_counter()
        result = optimize_training_batch(
            trainer,
            state,
            global_step,
            start,
        )
        persist_training_batch(
            state,
            global_step,
            result,
            step_started,
        )

    state["metrics_document"]["summary"]["completed"] = True
    state["metrics_document"]["summary"]["total_time_s"] = (
        time.perf_counter() - state["training_started"]
    )
    state["gae_document"]["summary"]["completed"] = True
    write_json(state["metrics_path"], state["metrics_document"])
    write_json(state["gae_path"], state["gae_document"], indent=None)
    state["policy"].save_pretrained(state["output_dir"])
    tokenizer.save_pretrained(state["output_dir"])
    state["run"].finish()
    return trainer.state

## Thin TRL trainer wrapper

In [ ]:
class ExactAnswerPPO2Trainer(PPOTrainer):
    def __init__(self, *args, exact_train_rows, **kwargs):
        super().__init__(*args, **kwargs)
        self.exact_train_rows = exact_train_rows

    def train(self):
        return run_ppo2_training(self)

## TRL dataset adapter and PPO configuration

In [ ]:
tokenized_prompts = train_rows.map(
    lambda row: tokenizer(
        prompt_text(row["question"]),
        truncation=True,
        max_length=max_prompt_tokens,
    ),
    remove_columns=train_rows.column_names,
)

ppo_args = PPOConfig(
    output_dir=str(project_root / output_path),
    run_name=wandb_run_name,
    sft_model_path=str(checkpoint_path),
    num_train_epochs=num_epochs,
    total_episodes=train_problems,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=1,
    local_rollout_forward_batch_size=batch_size,
    response_length=max_completion_tokens,
    stop_token="eos",
    temperature=temperature,
    learning_rate=actor_learning_rate,
    num_ppo_epochs=num_ppo_epochs,
    num_mini_batches=1,
    num_sample_generations=0,
    cliprange=policy_clip_epsilon,
    cliprange_value=value_clip_epsilon,
    kl_coef=kl_coefficient,
    gamma=gamma,
    lam=gae_lambda,
    max_grad_norm=max_grad_norm,
    seed=seed,
    report_to=[],
    bf16=True,
)

## Trainer construction

In [ ]:
trainer = ExactAnswerPPO2Trainer(
    args=ppo_args,
    processing_class=tokenizer,
    model=actor,
    ref_model=reference,
    reward_model=unused_reward_model,
    train_dataset=tokenized_prompts,
    value_model=critic,
    exact_train_rows=train_rows,
)

## Run training with persistent failure metadata

In [ ]:
try:
    trainer.train()
except Exception as error:
    failure_output_dir = (project_root / output_path).resolve()
    failure_output_dir.mkdir(parents=True, exist_ok=True)
    for filename in ("metrics.json", "gae.json"):
        path = failure_output_dir / filename
        document = json.loads(path.read_text()) if path.exists() else {}
        document["summary"] = {
            **document.get("summary", {}),
            "completed": False,
            "error_type": type(error).__name__,
            "error_message": str(error),
        }
        write_json(path, document, indent=None if filename == "gae.json" else 2)
    if wandb.run is not None:
        wandb.finish(exit_code=1)
    torch.cuda.empty_cache()
    raise